# Phase 3 — DSPy Optimization + QLoRA Fine-Tuning (Tier 1, N=50)
Compares against Phase 2b's N=10 results (DSPy 100%, QLoRA 100%, both on held-out).

All fixes from Phase 2b are now baked into the code permanently (deepcopy patch, device_map fix, lightweight MIPROv2 settings) — this notebook should run straight through without the debugging Phase 2b needed.

**Before running:** Runtime -> Change runtime type -> T4 GPU. Do this BEFORE running any cell below.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!nvidia-smi

**STOP: must show a Tesla T4 GPU table with 0MiB used.** If not, fix runtime type and reconnect before continuing.

## 1. Clone repo (safe to re-run any time)

In [ ]:
import shutil, os
os.chdir("/content")
if os.path.exists("agentic-prompt-vs-finetune"):
    shutil.rmtree("agentic-prompt-vs-finetune")
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
!pwd

In [ ]:
!grep -A3 "def optimize" dspy_optimize.py

This should show `def optimize(lm, training_examples: list, max_new_tokens: int = 150):` — confirming the permanently-fixed version pulled in correctly.

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets dspy-ai optuna

In [ ]:
from huggingface_hub import login
login()

## 2. Sample the N=50 training set

In [ ]:
import sys, json
sys.path.insert(0, ".")
from envs.training_data import sample_tier1_training

train_50 = sample_tier1_training(50)
print(f"Sampled {len(train_50)} training examples for N=50 regime.")
print("First 3:", train_50[:3])

## 3. Load model + run DSPy optimization

In [ ]:
from envs.agent_harness import load_model
from envs.dspy_lm import LocalLlamaLM
from tasks.tier1 import TIER1_HELDOUT
import dspy_optimize

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
lm = LocalLlamaLM(model, tok)
print("Model + DSPy LM wrapper ready.")

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

optimized_program = dspy_optimize.optimize(lm, train_50)
print("DSPy optimization complete.")

N=50 means more bootstrapping candidates to try (still capped at num_trials=3 and 1 demo each, per the lightweight settings), so this may take somewhat longer than the N=10 run did. If it OOMs anyway despite the fixes, don't improvise new settings live — paste the full error here first.

In [ ]:
dspy_results = dspy_optimize.evaluate_program(optimized_program, TIER1_HELDOUT)
for r in dspy_results:
    print(f"[{'PASS' if r['grade']['success'] else 'FAIL'}] {r['id']} — {r['grade'].get('failure_type')}")

dspy_success_rate = sum(r["grade"]["success"] for r in dspy_results) / len(dspy_results)
print(f"\nDSPy-optimized (N=50) held-out success rate: {dspy_success_rate:.1%}")

with open("results/tier1_dspy_n50_results.json", "w") as f:
    json.dump(dspy_results, f, indent=2)

## 4. Download DSPy results NOW, before continuing to QLoRA
Do this immediately — don't wait, in case the session drops before QLoRA finishes.

In [ ]:
from google.colab import files
files.download("results/tier1_dspy_n50_results.json")

## 5. Free GPU memory, then restart runtime before QLoRA
Recommended based on today's experience: restart the runtime completely (Runtime -> Restart session, or Disconnect and delete runtime if issues persist) rather than trying to free memory in-place. Then re-run cells 1-2 (env var, GPU check) and Section 1 (clone/install/login) again before continuing to Section 6.

## 6. QLoRA fine-tuning (N=50)

In [ ]:
!python qlora_finetune.py --n 50

Watch the training loss — should trend downward across steps, same pattern as N=10's run (4.1 -> 1.68). With 50 examples instead of 10, expect more training steps and a longer runtime.

## 7. Evaluate the fine-tuned adapter

In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from envs.agent_harness import run_agent
from envs.tools import TOOL_SCHEMAS, call_tool
from tasks.tier1 import TIER1_HELDOUT
from grader import grade_task

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", quantization_config=bnb_config, device_map={"": 0})
ft_model = PeftModel.from_pretrained(base_model, "adapters/tier1_n50")
ft_tok = AutoTokenizer.from_pretrained("adapters/tier1_n50")
print("Fine-tuned model loaded.")

In [ ]:
qlora_results = []
for task in TIER1_HELDOUT:
    tool_calls, final_text = run_agent(ft_model, ft_tok, task["prompt"], TOOL_SCHEMAS, call_tool)
    grade = grade_task(task, tier=1, tool_calls=tool_calls, final_text=final_text)
    qlora_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

qlora_success_rate = sum(r["grade"]["success"] for r in qlora_results) / len(qlora_results)
print(f"\nQLoRA fine-tuned (N=50) held-out success rate: {qlora_success_rate:.1%}")

with open("results/tier1_qlora_n50_results.json", "w") as f:
    json.dump(qlora_results, f, indent=2)

## 8. Download everything and push

In [ ]:
from google.colab import files
files.download("results/tier1_qlora_n50_results.json")
files.download("adapters/tier1_n50/training_examples.json")

Move both downloaded files into `results/` on your laptop (rename `training_examples.json` to `training_examples_n50.json`), then:
```bash
git add results/tier1_dspy_n50_results.json results/tier1_qlora_n50_results.json results/training_examples_n50.json
git commit -m "Phase 3: DSPy + QLoRA results for Tier 1, N=50"
git push
```
(If you already downloaded and pushed the DSPy N=50 result back in Section 4, just add the QLoRA files here.)